# Week 4 Lab: Regularization for Deep Learning

**Course:** Deep Learning (COE 443) — Istinye University

**Instructor:** Asst. Prof. Dr. Yigit Bekir Kaya

**Reference:** Goodfellow, Bengio, Courville — *Deep Learning*, Chapter 7

---

## Objectives

In this lab you will:

1. **See overfitting in action** — train an overparameterized MLP and watch the train/test gap grow
2. **Implement L2 regularization from scratch** using NumPy and see how weight decay shrinks weights
3. **Implement L1 regularization** and observe sparsity — weights driven to exactly zero
4. **Build dropout from scratch** — create random binary masks and verify against PyTorch
5. **Implement early stopping** with a patience-based algorithm (Algorithm 7.1)
6. **Apply data augmentation and label smoothing** to improve generalization
7. **Use batch normalization** and see its dual role: faster training + regularization
8. **Compare all techniques** side by side on Fashion-MNIST

By the end, you will understand **why** each regularization technique works, **how** to implement them, and **when** to use each one in practice.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from copy import deepcopy

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Load Fashion-MNIST

Fashion-MNIST is a drop-in replacement for MNIST with 10 clothing classes. It is significantly harder than digit MNIST and better demonstrates the effects of regularization.

**Classes:** T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

In [ ]:
# Download Fashion-MNIST
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_basic)
test_dataset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_basic)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=256, shuffle=False)

# Also load as raw NumPy arrays (for from-scratch parts)
raw_train = datasets.FashionMNIST(root='./data', train=True,  download=True)
raw_test  = datasets.FashionMNIST(root='./data', train=False, download=True)

X_train_np = raw_train.data.numpy().reshape(-1, 784).astype(np.float64) / 255.0
y_train_np = raw_train.targets.numpy()
X_test_np  = raw_test.data.numpy().reshape(-1, 784).astype(np.float64)  / 255.0
y_test_np  = raw_test.targets.numpy()

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal',  'Shirt',   'Sneaker',  'Bag',   'Ankle boot']

print(f"Training: {X_train_np.shape[0]:,} images  |  Test: {X_test_np.shape[0]:,} images")
print(f"Each image: {X_train_np.shape[1]} pixels (28x28 flattened)")
print(f"Classes: {class_names}")

# Visualize samples
fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
for cls in range(10):
    idx = np.where(y_train_np == cls)[0][0]
    axes[0, cls].imshow(X_train_np[idx].reshape(28, 28), cmap='gray')
    axes[0, cls].set_title(class_names[cls], fontsize=9)
    axes[0, cls].axis('off')
    idx2 = np.where(y_train_np == cls)[0][1]
    axes[1, cls].imshow(X_train_np[idx2].reshape(28, 28), cmap='gray')
    axes[1, cls].axis('off')
plt.suptitle('Fashion-MNIST: Two samples per class', fontsize=13)
plt.tight_layout()
plt.show()

---

# Part 1: Overfitting in Action

**From the lecture (Slides 1–5):** Overfitting occurs when a model learns the training data so well that it fails to generalize. An overparameterized model has so many weights that it can memorize training samples instead of learning general patterns.

We train a deliberately large MLP (784 → 512 → 256 → 128 → 10) for 50 epochs and observe the growing gap between training and test performance.

In [ ]:
class OverparameterizedMLP(nn.Module):
    """Deliberately large MLP to demonstrate overfitting."""
    def __init__(self, dropout_p=0.0, use_bn=False):
        super().__init__()
        self.use_bn = use_bn
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(p=dropout_p)
        if use_bn:
            self.bn1 = nn.BatchNorm1d(512)
            self.bn2 = nn.BatchNorm1d(256)
            self.bn3 = nn.BatchNorm1d(128)

    def forward(self, x):
        x = self.fc1(x)
        if self.use_bn:
            x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        if self.use_bn:
            x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.fc3(x)
        if self.use_bn:
            x = self.bn3(x)
        x = F.relu(x)
        x = self.dropout(x)

        return self.fc4(x)


n_params = sum(p.numel() for p in OverparameterizedMLP().parameters())
print(f"Model parameters: {n_params:,}")
print(f"Training samples: {len(train_dataset):,}")
print(f"Parameter-to-sample ratio: {n_params / len(train_dataset):.1f}x  (overparam!)")

In [ ]:
def train_model(model, train_loader, test_loader, epochs=50,
                lr=0.001, weight_decay=0.0, l1_lambda=0.0,
                verbose=True, use_augmented_loader=None):
    """General training loop. Returns dicts with per-epoch metrics."""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    loader = use_augmented_loader if use_augmented_loader is not None else train_loader

    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for X_b, y_b in loader:
            X_b = X_b.view(-1, 784)
            logits = model(X_b)
            loss = criterion(logits, y_b)

            # Optional L1 penalty
            if l1_lambda > 0:
                l1_norm = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_norm

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y_b.size(0)
            correct += (logits.argmax(1) == y_b).sum().item()
            total += y_b.size(0)

        history['train_loss'].append(total_loss / total)
        history['train_acc'].append(correct / total)

        # --- Evaluation ---
        model.eval()
        test_loss, test_correct, test_total = 0.0, 0, 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b = X_b.view(-1, 784)
                logits = model(X_b)
                loss = criterion(logits, y_b)
                test_loss += loss.item() * y_b.size(0)
                test_correct += (logits.argmax(1) == y_b).sum().item()
                test_total += y_b.size(0)

        history['test_loss'].append(test_loss / test_total)
        history['test_acc'].append(test_correct / test_total)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Train loss: {history['train_loss'][-1]:.4f} acc: {history['train_acc'][-1]:.3f} | "
                  f"Test  loss: {history['test_loss'][-1]:.4f} acc: {history['test_acc'][-1]:.3f}")

    return history

In [ ]:
# Train the overparameterized model for 50 epochs — no regularization
torch.manual_seed(42)
baseline_model = OverparameterizedMLP()

print("Training overparameterized MLP (no regularization) for 50 epochs...")
baseline_history = train_model(baseline_model, train_loader, test_loader, epochs=50)

In [ ]:
epochs_range = range(1, 51)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, baseline_history['train_loss'], label='Train loss', color='royalblue')
axes[0].plot(epochs_range, baseline_history['test_loss'],  label='Test loss',  color='tomato')
axes[0].fill_between(epochs_range,
                     baseline_history['train_loss'],
                     baseline_history['test_loss'],
                     alpha=0.15, color='tomato', label='Generalization gap')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Train vs Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a * 100 for a in baseline_history['train_acc']], label='Train acc', color='royalblue')
axes[1].plot(epochs_range, [a * 100 for a in baseline_history['test_acc']],  label='Test acc',  color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Train vs Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Overfitting in Action: Overparameterized MLP on Fashion-MNIST', fontsize=13)
plt.tight_layout()
plt.show()

final_train_acc = baseline_history['train_acc'][-1] * 100
final_test_acc  = baseline_history['test_acc'][-1]  * 100
print(f"\nEpoch 50 — Train accuracy: {final_train_acc:.1f}%  |  Test accuracy: {final_test_acc:.1f}%")
print(f"Generalization gap: {final_train_acc - final_test_acc:.1f} percentage points")
print("\nObservations:")
print("  - Train loss keeps decreasing towards 0 (memorization)")
print("  - Test loss diverges after an initial decrease (overfitting!)")
print("  - Train accuracy approaches 100% while test accuracy plateaus")

---

# Part 2: L2 Regularization (Weight Decay) From Scratch

**From the lecture (Slides 6–10, Eq. 7.1–7.5, Figure 7.1):**

L2 regularization adds a penalty on the squared norm of the weights to the objective:

$$\tilde{J}(\mathbf{w}; \mathbf{X}, \mathbf{y}) = J(\mathbf{w}; \mathbf{X}, \mathbf{y}) + \frac{\alpha}{2}\mathbf{w}^\top\mathbf{w}$$

The gradient becomes:

$$\nabla_\mathbf{w} \tilde{J} = \alpha\mathbf{w} + \nabla_\mathbf{w} J$$

Substituting into SGD: $\mathbf{w} \leftarrow \mathbf{w} - \epsilon(\alpha\mathbf{w} + \nabla_\mathbf{w} J)$, which rearranges to:

$$\mathbf{w} \leftarrow (1 - \epsilon\alpha)\mathbf{w} - \epsilon\nabla_\mathbf{w} J$$

Every update **shrinks the weights** by factor $(1 - \epsilon\alpha)$ before adding the gradient step — this is why L2 is called **weight decay**.

In [ ]:
# ----- NumPy helper functions (from-scratch) -----

def softmax_np(z):
    z_shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def forward_np(X, params):
    """2-layer MLP forward pass. Returns (y_hat, cache)."""
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    a1 = X @ W1 + b1
    h1 = np.maximum(0, a1)
    a2 = h1 @ W2 + b2
    y_hat = softmax_np(a2)
    return y_hat, (X, a1, h1)


def ce_loss_np(y_hat, y_true, params, alpha_l2=0.0, alpha_l1=0.0):
    """Cross-entropy loss with optional L1/L2 penalties."""
    m = y_true.shape[0]
    ce = -np.mean(np.log(y_hat[np.arange(m), y_true] + 1e-12))
    l2 = 0.0
    l1 = 0.0
    if alpha_l2 > 0:
        l2 = (alpha_l2 / 2) * (np.sum(params['W1']**2) + np.sum(params['W2']**2))
    if alpha_l1 > 0:
        l1 = alpha_l1 * (np.sum(np.abs(params['W1'])) + np.sum(np.abs(params['W2'])))
    return ce + l2 + l1


def backward_np(y_hat, y_true, cache, params, alpha_l2=0.0, alpha_l1=0.0):
    """Backprop through 2-layer MLP with optional L1/L2 gradient terms."""
    X, a1, h1 = cache
    W1, W2 = params['W1'], params['W2']
    m = y_true.shape[0]

    y_oh = np.zeros_like(y_hat)
    y_oh[np.arange(m), y_true] = 1.0
    delta2 = (y_hat - y_oh) / m          # (m, 10)

    dW2 = h1.T @ delta2 + alpha_l2 * W2 + alpha_l1 * np.sign(W2)
    db2 = delta2.sum(axis=0)

    dh1    = delta2 @ W2.T
    delta1 = dh1 * (a1 > 0)

    dW1 = X.T @ delta1 + alpha_l2 * W1 + alpha_l1 * np.sign(W1)
    db1 = delta1.sum(axis=0)

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}


def init_params(n_in=784, n_hidden=128, n_out=10, seed=42):
    np.random.seed(seed)
    return {
        'W1': np.random.randn(n_in, n_hidden) * np.sqrt(2.0 / n_in),
        'b1': np.zeros(n_hidden),
        'W2': np.random.randn(n_hidden, n_out) * np.sqrt(2.0 / n_hidden),
        'b2': np.zeros(n_out),
    }


def train_numpy_mlp(X_train, y_train, X_test, y_test,
                    epochs=20, lr=0.1, batch_size=128,
                    alpha_l2=0.0, alpha_l1=0.0, verbose=True):
    params = init_params()
    m = X_train.shape[0]
    history = {'train_loss': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        perm = np.random.permutation(m)
        Xs, ys = X_train[perm], y_train[perm]
        epoch_loss = 0.0
        n_batches = 0

        for start in range(0, m, batch_size):
            Xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]

            y_hat, cache = forward_np(Xb, params)
            loss = ce_loss_np(y_hat, yb, params, alpha_l2, alpha_l1)
            grads = backward_np(y_hat, yb, cache, params, alpha_l2, alpha_l1)

            for key in ['W1', 'b1', 'W2', 'b2']:
                params[key] -= lr * grads[f'd{key}']

            epoch_loss += loss
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        history['train_loss'].append(avg_loss)

        y_hat_test, _ = forward_np(X_test, params)
        test_acc = np.mean(y_hat_test.argmax(axis=1) == y_test)
        history['test_acc'].append(test_acc)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"Epoch {epoch:2d}/{epochs} | Loss: {avg_loss:.4f} | Test Acc: {test_acc:.3f}")

    return params, history


print("NumPy helper functions defined.")

In [ ]:
# Train with and without L2 regularization
print("Training WITHOUT L2 regularization...")
params_no_reg, hist_no_reg = train_numpy_mlp(
    X_train_np, y_train_np, X_test_np, y_test_np,
    epochs=20, lr=0.1, alpha_l2=0.0
)

print("\nTraining WITH L2 regularization (alpha=0.001)...")
params_l2, hist_l2 = train_numpy_mlp(
    X_train_np, y_train_np, X_test_np, y_test_np,
    epochs=20, lr=0.1, alpha_l2=0.001
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(hist_no_reg['train_loss'], label='No regularization', color='tomato')
axes[0].plot(hist_l2['train_loss'],     label='L2 (alpha=0.001)',  color='royalblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Weight histograms — W1 first layer
axes[1].hist(params_no_reg['W1'].ravel(), bins=60, alpha=0.6, label='No regularization', color='tomato', density=True)
axes[1].hist(params_l2['W1'].ravel(),     bins=60, alpha=0.6, label='L2 (alpha=0.001)', color='royalblue', density=True)
axes[1].set_xlabel('Weight value')
axes[1].set_ylabel('Density')
axes[1].set_title('W1 Weight Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Effect of different alpha values on weight magnitudes
alphas = [0.0, 0.0001, 0.001, 0.01, 0.1]
mean_magnitudes = []
for a in alphas:
    np.random.seed(42)
    p, _ = train_numpy_mlp(X_train_np, y_train_np, X_test_np, y_test_np,
                           epochs=10, lr=0.1, alpha_l2=a, verbose=False)
    mean_magnitudes.append(np.mean(np.abs(p['W1'])))

axes[2].plot(range(len(alphas)), mean_magnitudes, 'o-', color='purple', markersize=8)
axes[2].set_xticks(range(len(alphas)))
axes[2].set_xticklabels([str(a) for a in alphas])
axes[2].set_xlabel('L2 alpha')
axes[2].set_ylabel('Mean |W1|')
axes[2].set_title('Weight Magnitude vs Alpha')
axes[2].grid(True, alpha=0.3)

plt.suptitle('L2 Regularization: Weight Decay Effect', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Mean |W1| without L2: {np.mean(np.abs(params_no_reg['W1'])):.4f}")
print(f"Mean |W1| with L2:    {np.mean(np.abs(params_l2['W1'])):.4f}")
print("\nL2 pushes weights toward (but not all the way to) zero.")

In [ ]:
# Verify: PyTorch weight_decay does exactly the same thing
torch.manual_seed(42)
model_l2_pt = OverparameterizedMLP()
optimizer_l2 = optim.Adam(model_l2_pt.parameters(), lr=0.001, weight_decay=0.001)

criterion = nn.CrossEntropyLoss()
model_l2_pt.train()
for X_b, y_b in train_loader:
    X_b = X_b.view(-1, 784)
    loss = criterion(model_l2_pt(X_b), y_b)
    optimizer_l2.zero_grad()
    loss.backward()
    optimizer_l2.step()
    break  # just one batch to illustrate

w1_norm = model_l2_pt.fc1.weight.data.abs().mean().item()
print("PyTorch Adam with weight_decay=0.001 applied one batch step.")
print(f"Mean |fc1 weights|: {w1_norm:.4f}")
print("\nThe `weight_decay` parameter in PyTorch optimizers IS L2 regularization.")
print("It adds alpha * w to every gradient before the update — exactly Eq. 7.2.")

---

# Part 3: L1 Regularization and Sparsity

**From the lecture (Slides 11–14, Eq. 7.18–7.23):**

L1 regularization adds the **sum of absolute values** of weights:

$$\Omega(\theta) = ||\mathbf{w}||_1 = \sum_i |w_i|$$

The regularized gradient becomes:

$$\nabla_\mathbf{w} \tilde{J} = \alpha \text{sign}(\mathbf{w}) + \nabla_\mathbf{w} J$$

**Key difference from L2:** The gradient of L1 is $\pm\alpha$ (constant) regardless of weight magnitude, while the gradient of L2 is $\alpha w$ (proportional to the weight). This means L1 can push weights all the way to **exactly zero**, creating **sparse** parameter vectors.

In [ ]:
# Visualize L1 vs L2 penalty surfaces and their gradients
w = np.linspace(-2, 2, 400)

l1_penalty = np.abs(w)
l2_penalty = w**2 / 2
l1_grad    = np.sign(w)
l2_grad    = w

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(w, l1_penalty, label='L1: |w|',   color='tomato',    lw=2)
axes[0].plot(w, l2_penalty, label='L2: w²/2',  color='royalblue', lw=2)
axes[0].axvline(0, color='gray', lw=1, linestyle='--')
axes[0].set_xlabel('w')
axes[0].set_ylabel('Penalty')
axes[0].set_title('Penalty Functions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(w, l1_grad, label='L1 gradient: sign(w)',  color='tomato',    lw=2)
axes[1].plot(w, l2_grad, label='L2 gradient: w',        color='royalblue', lw=2)
axes[1].axhline(0, color='gray', lw=1)
axes[1].axvline(0, color='gray', lw=1, linestyle='--')
axes[1].set_xlabel('w')
axes[1].set_ylabel('Gradient')
axes[1].set_title('Gradient of Penalty')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-2, 2)

plt.suptitle('L1 vs L2: Why L1 Creates Sparsity', fontsize=13)
plt.tight_layout()
plt.show()

print("Key insight:")
print("  L2 gradient = w  → small weights get tiny nudges, never reach exactly 0")
print("  L1 gradient = ±1 → every nonzero weight gets a constant push toward 0")
print("  L1 can drive weights to EXACTLY zero; L2 only approaches zero asymptotically")

In [ ]:
# Train with L1 regularization from scratch
print("Training WITH L1 regularization (alpha=0.0005)...")
params_l1, hist_l1 = train_numpy_mlp(
    X_train_np, y_train_np, X_test_np, y_test_np,
    epochs=20, lr=0.1, alpha_l1=0.0005
)

In [ ]:
# Count exact zeros (or near-zeros, threshold=1e-4)
threshold = 1e-4

def count_near_zeros(params, threshold=1e-4):
    w1_zeros = np.sum(np.abs(params['W1']) < threshold)
    w2_zeros = np.sum(np.abs(params['W2']) < threshold)
    total    = params['W1'].size + params['W2'].size
    return w1_zeros + w2_zeros, total

nz_none, total = count_near_zeros(params_no_reg)
nz_l2,   _     = count_near_zeros(params_l2)
nz_l1,   _     = count_near_zeros(params_l1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Weight histograms
for ax, params, label, color in [
    (axes[0], params_no_reg, 'No Reg',    'gray'),
    (axes[1], params_l2,     'L2 Reg',    'royalblue'),
    (axes[2], params_l1,     'L1 Reg',    'tomato'),
]:
    ax.hist(params['W1'].ravel(), bins=80, color=color, alpha=0.7, density=True)
    ax.set_xlabel('Weight value')
    ax.set_ylabel('Density')
    ax.set_title(f'W1 Distribution — {label}')
    ax.axvline(0, color='black', lw=1.5, linestyle='--')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.5, 0.5)

plt.suptitle('Weight Distributions: No Reg vs L2 vs L1', fontsize=13)
plt.tight_layout()
plt.show()

# Bar chart: zero weights
labels_bar = ['No Reg', 'L2', 'L1']
zero_counts = [nz_none, nz_l2, nz_l1]
colors_bar  = ['gray', 'royalblue', 'tomato']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels_bar, zero_counts, color=colors_bar, alpha=0.8)
for bar, count in zip(bars, zero_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}\n({count/total:.1%})', ha='center', fontsize=11)
ax.set_ylabel('Near-zero weights (|w| < 1e-4)')
ax.set_title(f'Sparsity: Number of Near-Zero Weights (total = {total:,})')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Near-zero weights (|w| < {threshold}):")
print(f"  No regularization: {nz_none:,} / {total:,} ({nz_none/total:.1%})")
print(f"  L2 regularization: {nz_l2:,} / {total:,} ({nz_l2/total:.1%})")
print(f"  L1 regularization: {nz_l1:,} / {total:,} ({nz_l1/total:.1%})")
print("\nL1 creates FAR more near-zero weights → sparse representations!")

---

# Part 4: Dropout From Scratch

**From the lecture (Slides 19–23, Figure 7.6):**

Dropout randomly sets each neuron's output to 0 with probability $p$ during training. This can be viewed as training an **exponential ensemble** of $2^n$ thinned subnetworks that all share weights.

**Inverted dropout** (the standard implementation): scale active neurons by $\frac{1}{1-p}$ during training so that expected activations are unchanged. No scaling is needed at test time.

$$E[h_{\text{drop}}] = (1-p) \cdot \frac{h}{1-p} + p \cdot 0 = h$$

In [ ]:
def dropout_forward(h, p, training=True):
    """Inverted dropout: scale by 1/(1-p) during training so test needs no change."""
    if not training or p == 0.0:
        return h, None
    mask = (np.random.rand(*h.shape) > p).astype(h.dtype) / (1.0 - p)
    return h * mask, mask


# Verify: E[h_dropped] == h
np.random.seed(0)
h_test = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
p_drop = 0.5
n_trials = 100_000

dropped_means = np.zeros_like(h_test)
for _ in range(n_trials):
    h_d, _ = dropout_forward(h_test.copy(), p_drop, training=True)
    dropped_means += h_d
dropped_means /= n_trials

print("Inverted Dropout — Expectation Check (p=0.5, 100,000 trials):")
print(f"  Original h:       {h_test}")
print(f"  E[h_dropped]:     {dropped_means}")
print(f"  Max difference:   {np.abs(h_test - dropped_means).max():.4f}")
print("\nInverted dropout preserves expected activation values!")

In [ ]:
# Visualize dropout at different rates
np.random.seed(1)
h_full = np.random.randn(5, 10)  # 5 examples, 10 hidden units

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
drop_rates = [0.0, 0.3, 0.5, 0.8]
for ax, p in zip(axes, drop_rates):
    h_d, mask = dropout_forward(h_full.copy(), p, training=True)
    active_pct = (h_d != 0).mean() * 100
    im = ax.imshow(h_d, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
    ax.set_title(f'Dropout p={p}\n{active_pct:.0f}% active')
    ax.set_xlabel('Hidden units')
    ax.set_ylabel('Examples')
plt.suptitle('Dropout Masks: White = zeroed neuron', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Train MLP with dropout vs without (NumPy from scratch)
def train_numpy_mlp_dropout(X_train, y_train, X_test, y_test,
                             epochs=20, lr=0.05, batch_size=128,
                             dropout_p=0.0, verbose=True):
    params = init_params()
    m = X_train.shape[0]
    history = {'train_loss': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        perm = np.random.permutation(m)
        Xs, ys = X_train[perm], y_train[perm]
        epoch_loss, n_batches = 0.0, 0

        for start in range(0, m, batch_size):
            Xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]

            # Forward with dropout on hidden layer
            W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
            a1 = Xb @ W1 + b1
            h1_raw = np.maximum(0, a1)
            h1, mask = dropout_forward(h1_raw, dropout_p, training=True)
            a2 = h1 @ W2 + b2
            y_hat = softmax_np(a2)

            loss = -np.mean(np.log(y_hat[np.arange(len(yb)), yb] + 1e-12))

            # Backward (mask the gradient too)
            y_oh = np.zeros_like(y_hat)
            y_oh[np.arange(len(yb)), yb] = 1.0
            delta2 = (y_hat - y_oh) / len(yb)

            dW2 = h1.T @ delta2
            db2 = delta2.sum(axis=0)

            dh1 = delta2 @ W2.T
            if mask is not None:
                dh1 = dh1 * mask        # apply same dropout mask to gradient
            delta1 = dh1 * (a1 > 0)

            dW1 = Xb.T @ delta1
            db1 = delta1.sum(axis=0)

            params['W1'] -= lr * dW1
            params['b1'] -= lr * db1
            params['W2'] -= lr * dW2
            params['b2'] -= lr * db2

            epoch_loss += loss
            n_batches  += 1

        history['train_loss'].append(epoch_loss / n_batches)

        # Evaluate (no dropout)
        y_hat_test, _ = forward_np(X_test, params)
        test_acc = np.mean(y_hat_test.argmax(axis=1) == y_test)
        history['test_acc'].append(test_acc)

        if verbose and epoch % 5 == 0:
            print(f"Epoch {epoch:2d}/{epochs} | Loss: {epoch_loss/n_batches:.4f} | Test Acc: {test_acc:.3f}")

    return params, history


print("Training WITHOUT dropout...")
_, hist_no_drop = train_numpy_mlp_dropout(
    X_train_np, y_train_np, X_test_np, y_test_np,
    epochs=20, lr=0.05, dropout_p=0.0
)

print("\nTraining WITH dropout (p=0.5)...")
_, hist_dropout = train_numpy_mlp_dropout(
    X_train_np, y_train_np, X_test_np, y_test_np,
    epochs=20, lr=0.05, dropout_p=0.5
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_np = range(1, 21)
axes[0].plot(epochs_np, hist_no_drop['train_loss'], label='No dropout',    color='tomato')
axes[0].plot(epochs_np, hist_dropout['train_loss'], label='Dropout p=0.5', color='royalblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_np, [a * 100 for a in hist_no_drop['test_acc']], label='No dropout',    color='tomato')
axes[1].plot(epochs_np, [a * 100 for a in hist_dropout['test_acc']], label='Dropout p=0.5', color='royalblue')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Dropout: NumPy From-Scratch vs PyTorch Verification', fontsize=13)
plt.tight_layout()
plt.show()

# Verify against PyTorch nn.Dropout
print("\nVerification: PyTorch nn.Dropout uses the same inverted-dropout algorithm.")
pt_drop = nn.Dropout(p=0.5)
pt_drop.train()
h_pt = torch.ones(1000) * 2.0
h_dropped_pt = pt_drop(h_pt)
print(f"  PyTorch: E[h_dropped] = {h_dropped_pt.mean().item():.4f}  (expected 2.0)")

---

# Part 5: Early Stopping (Algorithm 7.1)

**From the lecture (Slides 24–27, Algorithm 7.1, Figures 7.3 and 7.4):**

Early stopping monitors validation loss and stops training when it stops improving, returning the model from the best checkpoint.

**Algorithm 7.1 (pseudocode):**
```
Initialize θ, θ* = θ, i = 0, j = 0, v* = ∞
while j < patience:
    update θ using gradient on training minibatch
    i += 1
    if i mod n_validate == 0:
        v = validation_loss(θ)
        if v < v*:
            θ* = θ, v* = v, i* = i, j = 0
        else:
            j += 1
return θ*, i*
```

**Connection to L2 (Figure 7.4):** Early stopping limits how far parameters travel from their initialization. Since networks are typically initialized near 0, this is equivalent to a soft L2 constraint.

In [ ]:
def train_with_early_stopping(model, train_loader, test_loader,
                               epochs=100, lr=0.001, patience=10, verbose=True):
    """Train with early stopping. Returns history and best_epoch."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_model_state = deepcopy(model.state_dict())
    best_epoch = 0
    patience_counter = 0

    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for X_b, y_b in train_loader:
            X_b = X_b.view(-1, 784)
            logits = model(X_b)
            loss = criterion(logits, y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss    += loss.item() * y_b.size(0)
            train_correct += (logits.argmax(1) == y_b).sum().item()
            train_total   += y_b.size(0)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b = X_b.view(-1, 784)
                logits = model(X_b)
                loss = criterion(logits, y_b)
                val_loss    += loss.item() * y_b.size(0)
                val_correct += (logits.argmax(1) == y_b).sum().item()
                val_total   += y_b.size(0)

        avg_train_loss = train_loss / train_total
        avg_val_loss   = val_loss   / val_total
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_correct / train_total)
        history['test_loss'].append(avg_val_loss)
        history['test_acc'].append(val_correct / val_total)

        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = deepcopy(model.state_dict())
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"Epoch {epoch:3d} | train_loss: {avg_train_loss:.4f} | "
                  f"val_loss: {avg_val_loss:.4f} | patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}. Best epoch: {best_epoch}.")
            break

    # Restore best model
    model.load_state_dict(best_model_state)
    return history, best_epoch


torch.manual_seed(42)
es_model = OverparameterizedMLP()
print("Training with early stopping (patience=10, max 100 epochs)...")
es_history, best_epoch = train_with_early_stopping(
    es_model, train_loader, test_loader, epochs=100, patience=10
)

In [ ]:
actual_epochs = len(es_history['train_loss'])
ep_range = range(1, actual_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, ylabel, title in [
    (axes[0], 'loss',  'Cross-Entropy Loss', 'Train vs Validation Loss'),
    (axes[1], 'acc',   'Accuracy (%)',        'Train vs Validation Accuracy'),
]:
    train_vals = es_history[f'train_{metric}']
    test_vals  = es_history[f'test_{metric}']
    if metric == 'acc':
        train_vals = [v * 100 for v in train_vals]
        test_vals  = [v * 100 for v in test_vals]

    ax.plot(ep_range, train_vals, label='Train',      color='royalblue')
    ax.plot(ep_range, test_vals,  label='Validation', color='tomato')
    ax.axvline(best_epoch, color='green', linestyle='--', lw=2,
               label=f'Best epoch: {best_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Early Stopping: Algorithm 7.1 in Action', fontsize=13)
plt.tight_layout()
plt.show()

best_test_acc  = es_history['test_acc'][best_epoch - 1]  * 100
final_test_acc = es_history['test_acc'][-1]              * 100
print(f"Best epoch ({best_epoch}): test accuracy = {best_test_acc:.1f}%")
print(f"Final epoch ({actual_epochs}): test accuracy = {final_test_acc:.1f}%")
print(f"Gain from early stopping: {best_test_acc - final_test_acc:+.1f} percentage points")
print("\nBy returning the best checkpoint, we avoid the overfitting that occurs in later epochs.")

---

# Part 6: Data Augmentation and Label Smoothing

**From the lecture (Slides 28–33, Section 7.4 and 7.5.1):**

**Data augmentation** creates artificial training examples by applying label-preserving transformations (rotations, flips, crops). It is one of the most effective regularization techniques because it directly addresses the root cause of overfitting: limited training data.

**Label smoothing** replaces hard one-hot targets with soft targets:
- Hard: $[0, 0, 1, 0, \ldots]$
- Soft: $[\frac{\varepsilon}{k-1}, \frac{\varepsilon}{k-1}, 1-\varepsilon, \frac{\varepsilon}{k-1}, \ldots]$

This prevents the model from being overconfident and improves calibration.

In [ ]:
# Define augmented transform
transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

# Visualize augmentation
raw_sample_dataset = datasets.FashionMNIST(root='./data', train=True, download=True,
                                            transform=transforms.ToTensor())
aug_sample_dataset  = datasets.FashionMNIST(root='./data', train=True, download=True,
                                             transform=transform_augmented)

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(16, 4))
for i in range(n_show):
    img_orig, label = raw_sample_dataset[i]
    axes[0, i].imshow(img_orig.squeeze(), cmap='gray')
    axes[0, i].set_title(class_names[label], fontsize=9)
    axes[0, i].axis('off')

    img_aug, _ = aug_sample_dataset[i]
    # Undo normalization for display
    img_disp = img_aug.squeeze().numpy()
    axes[1, i].imshow(img_disp, cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Augmented', fontsize=10)
plt.suptitle('Data Augmentation: Random Flip + Rotation + Translation', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Create augmented data loader
aug_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True,
                                           transform=transform_augmented)
aug_train_loader  = torch.utils.data.DataLoader(aug_train_dataset, batch_size=128, shuffle=True)

# Train: baseline vs augmentation vs label smoothing vs both
torch.manual_seed(42)
model_baseline_p6 = OverparameterizedMLP()
print("[1/4] Training baseline (no augmentation, hard labels)...")
hist_baseline_p6 = train_model(model_baseline_p6, train_loader, test_loader,
                                epochs=30, verbose=False)
print(f"  Final test acc: {hist_baseline_p6['test_acc'][-1]:.3f}")

torch.manual_seed(42)
model_augment = OverparameterizedMLP()
print("[2/4] Training with data augmentation...")
hist_augment = train_model(model_augment, train_loader, test_loader,
                            epochs=30, verbose=False,
                            use_augmented_loader=aug_train_loader)
print(f"  Final test acc: {hist_augment['test_acc'][-1]:.3f}")

# Label smoothing via PyTorch's built-in
def train_label_smooth(model, loader, test_loader, epochs=30, lr=0.001, label_smooth=0.1):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    history = {'test_acc': []}
    for epoch in range(epochs):
        model.train()
        for X_b, y_b in loader:
            X_b = X_b.view(-1, 784)
            loss = criterion(model(X_b), y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b = X_b.view(-1, 784)
                correct += (model(X_b).argmax(1) == y_b).sum().item()
                total += y_b.size(0)
        history['test_acc'].append(correct / total)
    return history

torch.manual_seed(42)
model_smooth = OverparameterizedMLP()
print("[3/4] Training with label smoothing (eps=0.1)...")
hist_smooth = train_label_smooth(model_smooth, train_loader, test_loader,
                                  epochs=30, label_smooth=0.1)
print(f"  Final test acc: {hist_smooth['test_acc'][-1]:.3f}")

torch.manual_seed(42)
model_both = OverparameterizedMLP()
print("[4/4] Training with augmentation + label smoothing...")
hist_both = train_label_smooth(model_both, aug_train_loader, test_loader,
                                epochs=30, label_smooth=0.1)
print(f"  Final test acc: {hist_both['test_acc'][-1]:.3f}")

In [ ]:
ep_range_30 = range(1, 31)
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(ep_range_30, [a * 100 for a in hist_baseline_p6['test_acc']], label='Baseline',             color='gray')
ax.plot(ep_range_30, [a * 100 for a in hist_augment['test_acc']],     label='Augmentation',         color='royalblue')
ax.plot(ep_range_30, [a * 100 for a in hist_smooth['test_acc']],      label='Label smoothing',      color='orange')
ax.plot(ep_range_30, [a * 100 for a in hist_both['test_acc']],        label='Augment + Smoothing',  color='green')

ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Data Augmentation and Label Smoothing on Fashion-MNIST')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Final test accuracies:")
print(f"  Baseline:                {hist_baseline_p6['test_acc'][-1]:.3f}")
print(f"  Data augmentation:       {hist_augment['test_acc'][-1]:.3f}")
print(f"  Label smoothing:         {hist_smooth['test_acc'][-1]:.3f}")
print(f"  Augmentation + Smoothing:{hist_both['test_acc'][-1]:.3f}")

---

# Part 7: Batch Normalization

**From the lecture (Slides 34–37):**

Batch normalization (Ioffe & Szegedy 2015) normalizes each mini-batch of activations:

$$\hat{x}^{(k)} = \frac{x^{(k)} - \mu_\mathcal{B}^{(k)}}{\sqrt{(\sigma_\mathcal{B}^{(k)})^2 + \varepsilon}}$$

then applies a learned affine transform: $y^{(k)} = \gamma^{(k)} \hat{x}^{(k)} + \beta^{(k)}$.

**Dual role:**
1. **Reduces internal covariate shift** — keeps activation distributions stable → allows higher learning rates → faster training
2. **Implicit regularization** — mini-batch statistics add noise to activations during training (similar to dropout)

In [ ]:
# Compare: baseline vs BN vs dropout vs BN+dropout
torch.manual_seed(42)
model_no_bn   = OverparameterizedMLP(dropout_p=0.0, use_bn=False)
model_bn_only = OverparameterizedMLP(dropout_p=0.0, use_bn=True)
model_do_only = OverparameterizedMLP(dropout_p=0.5, use_bn=False)
model_bn_do   = OverparameterizedMLP(dropout_p=0.5, use_bn=True)

configs = [
    (model_no_bn,   'No BN, No Dropout',    'gray'),
    (model_bn_only, 'BN only',              'royalblue'),
    (model_do_only, 'Dropout only (p=0.5)', 'tomato'),
    (model_bn_do,   'BN + Dropout',         'green'),
]

bn_histories = {}
for model, label, color in configs:
    print(f"Training: {label}...")
    h = train_model(model, train_loader, test_loader, epochs=30, verbose=False)
    bn_histories[label] = (h, color)
    print(f"  Final test acc: {h['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for label, (h, color) in bn_histories.items():
    axes[0].plot(ep_range_30, h['train_loss'], color=color, label=label, alpha=0.9)
    axes[1].plot(ep_range_30, [a * 100 for a in h['test_acc']], color=color, label=label, alpha=0.9)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Loss')
axes[0].set_title('Training Loss')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Batch Normalization: Training Speed + Regularization', fontsize=13)
plt.tight_layout()
plt.show()

print("Observations:")
print("  - BN alone reduces training loss faster (internal covariate shift reduction)")
print("  - Dropout alone improves generalization but trains slower")
print("  - BN + Dropout combines both effects: fast convergence + good generalization")

---

# Part 8: Grand Comparison on Fashion-MNIST

We now train 7 models side by side using the same overparameterized architecture, Adam optimizer, and 30 epochs, varying only the regularization technique.

In [ ]:
# L1 via manual PyTorch training loop (model + l1_lambda argument)
# All others via train_model which supports weight_decay, dropout, augmentation

results = {}

# 1. Baseline
torch.manual_seed(42)
m1 = OverparameterizedMLP()
print("[1/7] Baseline (no regularization)...")
results['1_Baseline'] = train_model(m1, train_loader, test_loader, epochs=30, verbose=False)
print(f"  Best test acc: {max(results['1_Baseline']['test_acc']):.3f}")

# 2. L2 (weight decay)
torch.manual_seed(42)
m2 = OverparameterizedMLP()
print("[2/7] L2 regularization (weight_decay=0.001)...")
results['2_L2'] = train_model(m2, train_loader, test_loader,
                               epochs=30, weight_decay=0.001, verbose=False)
print(f"  Best test acc: {max(results['2_L2']['test_acc']):.3f}")

# 3. L1 (manual)
torch.manual_seed(42)
m3 = OverparameterizedMLP()
print("[3/7] L1 regularization (lambda=0.0001)...")
results['3_L1'] = train_model(m3, train_loader, test_loader,
                               epochs=30, l1_lambda=0.0001, verbose=False)
print(f"  Best test acc: {max(results['3_L1']['test_acc']):.3f}")

# 4. Dropout
torch.manual_seed(42)
m4 = OverparameterizedMLP(dropout_p=0.5)
print("[4/7] Dropout (p=0.5)...")
results['4_Dropout'] = train_model(m4, train_loader, test_loader, epochs=30, verbose=False)
print(f"  Best test acc: {max(results['4_Dropout']['test_acc']):.3f}")

# 5. Early stopping
torch.manual_seed(42)
m5 = OverparameterizedMLP()
print("[5/7] Early stopping (patience=5)...")
es_hist_cmp, es_best_cmp = train_with_early_stopping(
    m5, train_loader, test_loader, epochs=100, patience=5, verbose=False
)
# Pad to 30 epochs for consistent plotting
pad_len = max(0, 30 - len(es_hist_cmp['test_acc']))
es_hist_cmp['test_acc'] = es_hist_cmp['test_acc'] + [es_hist_cmp['test_acc'][-1]] * pad_len
results['5_EarlyStopping'] = es_hist_cmp
print(f"  Best test acc: {max(results['5_EarlyStopping']['test_acc']):.3f}  (stopped at epoch {es_best_cmp})")

# 6. Data augmentation
torch.manual_seed(42)
m6 = OverparameterizedMLP()
print("[6/7] Data augmentation...")
results['6_Augmentation'] = train_model(m6, train_loader, test_loader,
                                         epochs=30, verbose=False,
                                         use_augmented_loader=aug_train_loader)
print(f"  Best test acc: {max(results['6_Augmentation']['test_acc']):.3f}")

# 7. All combined: L2 + Dropout + Augmentation + BN
torch.manual_seed(42)
m7 = OverparameterizedMLP(dropout_p=0.5, use_bn=True)
print("[7/7] All combined (L2 + Dropout + Augmentation + BN)...")
results['7_Combined'] = train_model(m7, train_loader, test_loader,
                                     epochs=30, weight_decay=0.001, verbose=False,
                                     use_augmented_loader=aug_train_loader)
print(f"  Best test acc: {max(results['7_Combined']['test_acc']):.3f}")

In [ ]:
palette = [
    ('1_Baseline',       'Baseline',               'gray'),
    ('2_L2',             'L2 (wd=0.001)',           '#3182BD'),
    ('3_L1',             'L1 (λ=0.0001)',           '#E6550D'),
    ('4_Dropout',        'Dropout (p=0.5)',         '#31A354'),
    ('5_EarlyStopping',  'Early Stopping',          '#756BB1'),
    ('6_Augmentation',   'Augmentation',            '#E7298A'),
    ('7_Combined',       'All Combined',            '#D95F02'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for key, label, color in palette:
    h = results[key]
    n_ep = min(30, len(h['test_acc']))
    axes[0].plot(range(1, n_ep + 1), [a * 100 for a in h['test_acc'][:n_ep]],
                 label=label, color=color, lw=1.8)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Test Accuracy over Epochs (all techniques)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Bar chart: best accuracy
labels_bar  = [label for _, label, _ in palette]
colors_bar  = [color for _, _, color in palette]
best_accs   = [max(results[key]['test_acc']) * 100 for key, _, _ in palette]

bars = axes[1].bar(range(len(labels_bar)), best_accs, color=colors_bar, alpha=0.85)
axes[1].set_xticks(range(len(labels_bar)))
axes[1].set_xticklabels(labels_bar, rotation=35, ha='right', fontsize=9)
axes[1].set_ylabel('Best Test Accuracy (%)')
axes[1].set_title('Best Test Accuracy per Technique')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, acc in zip(bars, best_accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 f'{acc:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Grand Comparison: All Regularization Techniques on Fashion-MNIST', fontsize=13)
plt.tight_layout()
plt.show()

print("\nSummary:")
for key, label, _ in palette:
    print(f"  {label:30s}: best = {max(results[key]['test_acc'])*100:.1f}%")

---

# Summary

In this lab you implemented and compared the major regularization techniques from Chapter 7:

| Part | What You Did | Key Concept |
|------|-------------|-------------|
| 1 | Demonstrated overfitting | Train/test gap with overparameterized model |
| 2 | Implemented L2 from scratch | Weight decay shrinks all weights toward (not to) zero |
| 3 | Implemented L1 from scratch | Sparsity — constant gradient drives weights to exactly zero |
| 4 | Built dropout from scratch | Inverted dropout preserves expected activations |
| 5 | Implemented early stopping | Algorithm 7.1 — save best model, stop on patience |
| 6 | Applied augmentation + smoothing | More data diversity, softer probability targets |
| 7 | Used batch normalization | Faster training (stable activations) + implicit regularization |
| 8 | Compared all techniques | No single best — combining techniques works best |

**Core insight:** Every regularization technique reduces the effective capacity of the model, but in different ways. L2 penalizes large weights. L1 creates sparsity. Dropout trains an ensemble. Early stopping limits the search. Augmentation expands the data distribution. BN stabilizes training dynamics.

---

# Homework Exercises

### Exercise 1: Elastic Net
Implement **elastic net** regularization — a linear combination of L1 and L2:

$$\Omega(\theta) = \alpha_1 ||\mathbf{w}||_1 + \frac{\alpha_2}{2}||\mathbf{w}||_2^2$$

Use the `backward_np` function (it already supports both). Set `alpha_l1=0.0003, alpha_l2=0.001` and compare the sparsity pattern (zero-weight count) to pure L1 and pure L2.

### Exercise 2: Dropout Rate Sweep
Try dropout rates `[0.1, 0.3, 0.5, 0.7, 0.9]` using the `OverparameterizedMLP` class. Train each for 30 epochs and plot test accuracy vs dropout rate. At what rate does performance peak? What happens at very high rates (e.g., 0.9)?

### Exercise 3: Weight Noise as Regularization
Section 7.5 describes adding Gaussian noise to weights during training. Implement this by modifying the training loop:
```python
# During training, for each forward pass:
for p in model.parameters():
    p.data.add_(torch.randn_like(p) * noise_std)
```
Try `noise_std` in `[0.001, 0.01, 0.1]` and compare with dropout. What are the trade-offs?

---

### Conceptual Questions

1. **Why does L2 shrink weights toward zero but rarely set them exactly to zero, while L1 does?** Hint: think about the gradient of each penalty as $w \to 0$.

2. **Why is dropout equivalent to training an exponential number of subnetworks?** With $n$ hidden units each dropped independently, how many distinct subnetworks are implied? Why do they all share the same weights?

3. **Early stopping is sometimes called 'the most beautiful regularizer.' Why is it connected to L2 regularization?** Hint: think about where parameters start (near zero via initialization) and how far gradient descent moves them. Consult Figure 7.4 in Goodfellow et al.

---

### Next Week

**Week 5: Optimization for Training Deep Models (Ch 8)**
- SGD with momentum and Nesterov momentum
- Adaptive learning rates: AdaGrad, RMSProp, Adam
- Parameter initialization strategies
- Learning rate scheduling (warmup, cosine annealing)
- Gradient clipping and second-order methods